# 03 — Transmission Mechanism

**Purpose**: Quantify the causal chain from GOV → Revenue → EBITDA → Stock price.  
**Outputs**: Exhibit 5 causal chain table, variance decomposition of GOV.  

See CLAUDE.md §12 for full spec.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import sys
sys.path.insert(0, '..')

from src.config import MASTER_DF_PATH, CRSP_EVENT_STUDY_PATH, CHART_STYLE, COLORS, OUTPUTS_FIGURES
from src.model_transmission import (
    regression_gov_to_revenue, regression_revenue_to_ebitda,
    regression_gov_surprise_to_car, variance_decomposition_gov,
)

plt.rcParams.update(CHART_STYLE)
df = pd.read_csv(MASTER_DF_PATH, parse_dates=['quarter_end_date'])

## 1. Regression 1: GOV Growth → Revenue Growth

In [ ]:
r1 = regression_gov_to_revenue(df)
# Expected β ≈ 0.90–0.95 (stable take rate)

## 2. Regression 2: Revenue Surprise → EBITDA Margin Change

In [ ]:
r2 = regression_revenue_to_ebitda(df)

## 3. Regression 3: GOV Surprise → CAR[-1,+2]

In [ ]:
r3 = regression_gov_surprise_to_car()

## 4. Variance Decomposition of GOV

In [ ]:
var_decomp = variance_decomposition_gov(df)
var_decomp

## 5. Exhibit 5 — Causal Chain Summary Table

In [ ]:
# Build the causal chain table for write-up
chain = pd.DataFrame([
    {'Link': 'Trends momentum → GOV surprise',
     'β / r': var_decomp.loc[var_decomp['col']=='doordash_trends_momentum', 'pct_variance_explained'].values[0] if not var_decomp.empty else 'n/a',
     'Metric': 'R² contribution (%)'},
    {'Link': 'GOV surprise → Revenue beat',
     'β / r': round(r1.params.get('gov_yoy_growth_pct', np.nan), 3) if r1 else 'n/a',
     'Metric': f'β (take-rate pass-through), R²={round(r1.rsquared, 2) if r1 else "n/a"}'},
    {'Link': 'Revenue surprise → EBITDA margin change',
     'β / r': round(r2.params.iloc[1], 3) if r2 else 'n/a',
     'Metric': f'β (operating leverage), R²={round(r2.rsquared, 2) if r2 else "n/a"}'},
    {'Link': 'GOV surprise → CAR[-1,+2]',
     'β / r': round(r3.params.get('gov_surprise_pct', np.nan), 3) if r3 else 'n/a',
     'Metric': f'β, R²={round(r3.rsquared, 2) if r3 else "n/a"}'},
])
print('\nExhibit 5 — Causal Chain (Data Signal → Stock Impact):')
print(chain.to_string(index=False))